![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/banner_1.png)

# Proyecto 2 - Clasificación de género de películas

El propósito de este proyecto es que puedan poner en práctica, en sus respectivos grupos de trabajo, sus conocimientos sobre técnicas de preprocesamiento, modelos predictivos de NLP, y la disponibilización de modelos. Para su desarrollo tengan en cuenta las instrucciones dadas en la "Guía del proyecto 2: Clasificación de género de películas"

**Entrega**: La entrega del proyecto deberán realizarla durante la semana 8. Sin embargo, es importante que avancen en la semana 7 en el modelado del problema y en parte del informe, tal y como se les indicó en la guía.

Para hacer la entrega, deberán adjuntar el informe autocontenido en PDF a la actividad de entrega del proyecto que encontrarán en la semana 8, y subir el archivo de predicciones a la [competencia de Kaggle](https://www.kaggle.com/t/29c44fce98c747f2a1dfdaf29d4c4965).

## Datos para la predicción de género en películas

![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/moviegenre.png)

En este proyecto se usará un conjunto de datos de géneros de películas. Cada observación contiene el título de una película, su año de lanzamiento, la sinopsis o plot de la película (resumen de la trama) y los géneros a los que pertenece (una película puede pertenercer a más de un género). Por ejemplo:
- Título: 'How to Be a Serial Killer'
- Plot: 'A serial killer decides to teach the secrets of his satisfying career to a video store clerk.'
- Generos: 'Comedy', 'Crime', 'Horror'

La idea es que usen estos datos para predecir la probabilidad de que una película pertenezca, dada la sinopsis, a cada uno de los géneros.

Agradecemos al profesor Fabio González, Ph.D. y a su alumno John Arevalo por proporcionar este conjunto de datos. Ver https://arxiv.org/abs/1702.01992

## Ejemplo predicción conjunto de test para envío a Kaggle
En esta sección encontrarán el formato en el que deben guardar los resultados de la predicción para que puedan subirlos a la competencia en Kaggle.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Importación librerías
import pandas as pd
import os
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, roc_auc_score
from sklearn.model_selection import train_test_split

In [ ]:
# Carga de datos de archivo .csv
dataTraining = pd.read_csv('https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTraining.zip', encoding='UTF-8', index_col=0)
dataTesting = pd.read_csv('https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTesting.zip', encoding='UTF-8', index_col=0)

In [ ]:
# Visualización datos de entrenamiento
dataTraining.head()

In [ ]:
# Visualización datos de test
dataTesting.head()

In [ ]:
# Definición de variables predictoras (X)
vect = CountVectorizer(max_features=1000)
X_dtm = vect.fit_transform(dataTraining['plot'])
X_dtm.shape

In [ ]:
# Definición de variable de interés (y)
dataTraining['genres'] = dataTraining['genres'].map(lambda x: eval(x))
le = MultiLabelBinarizer()
y_genres = le.fit_transform(dataTraining['genres'])

In [ ]:
# Separación de variables predictoras (X) y variable de interés (y) en set de entrenamiento y test usandola función train_test_split
X_train, X_test, y_train_genres, y_test_genres = train_test_split(X_dtm, y_genres, test_size=0.33, random_state=42)

In [ ]:
# Definición y entrenamiento
clf = OneVsRestClassifier(RandomForestClassifier(n_jobs=-1, n_estimators=100, max_depth=10, random_state=42))
clf.fit(X_train, y_train_genres)

In [ ]:
# Predicción del modelo de clasificación
y_pred_genres = clf.predict_proba(X_test)

# Impresión del desempeño del modelo
roc_auc_score(y_test_genres, y_pred_genres, average='macro')

In [ ]:
# transformación variables predictoras X del conjunto de test
X_test_dtm = vect.transform(dataTesting['plot'])

cols = ['p_Action', 'p_Adventure', 'p_Animation', 'p_Biography', 'p_Comedy', 'p_Crime', 'p_Documentary', 'p_Drama', 'p_Family',
        'p_Fantasy', 'p_Film-Noir', 'p_History', 'p_Horror', 'p_Music', 'p_Musical', 'p_Mystery', 'p_News', 'p_Romance',
        'p_Sci-Fi', 'p_Short', 'p_Sport', 'p_Thriller', 'p_War', 'p_Western']

# Predicción del conjunto de test
y_pred_test_genres = clf.predict_proba(X_test_dtm)

In [ ]:
# Guardar predicciones en formato exigido en la competencia de kaggle
res = pd.DataFrame(y_pred_test_genres, index=dataTesting.index, columns=cols)
res.to_csv('pred_genres_text_RF.csv', index_label='ID')
res.head()

### Desarrollo del proyecto

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from scipy.sparse import hstack, csr_matrix
from scipy.optimize import minimize

# ── Cargar datos ───────────────────────────────────────────────
dataTraining = pd.read_csv('https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTraining.zip', encoding='UTF-8', index_col=0)
dataTesting  = pd.read_csv('https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTesting.zip',  encoding='UTF-8', index_col=0)

# ── Limpieza ───────────────────────────────────────────────────
dataTraining1 = dataTraining.drop_duplicates(subset=['title', 'year'])
dataTraining1 = dataTraining1[dataTraining1['plot'].str.split().str.len() >= 10]
dataTraining1['genres'] = dataTraining1['genres'].map(lambda x: eval(x) if isinstance(x, str) else x)
print(f"Shape: {dataTraining1.shape}")

# ── Etiquetas ──────────────────────────────────────────────────
le       = MultiLabelBinarizer()
y_genres = le.fit_transform(dataTraining1['genres'])

# ── Split ──────────────────────────────────────────────────────
plots = dataTraining1['plot'].tolist()
idx_train, idx_test = train_test_split(range(len(plots)), test_size=0.33, random_state=42)
y_train = y_genres[idx_train]
y_test  = y_genres[idx_test]
print(f"Train: {len(idx_train)} | Test: {len(idx_test)}")

# ── Cargar embeddings guardados ────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

X_bert_large = np.load('/content/drive/MyDrive/ML/X_bert_large.npy')
X_roberta    = np.load('/content/drive/MyDrive/ML/X_roberta.npy')
X_e5         = np.load('/content/drive/MyDrive/ML/X_e5.npy')
print(f"Embeddings cargados ✓")

# ── TF-IDF ─────────────────────────────────────────────────────
vect_tfidf = TfidfVectorizer(max_features=10000, stop_words='english')
X_tfidf    = vect_tfidf.fit_transform(dataTraining1['plot']).astype(np.float32)

# ── Splits ─────────────────────────────────────────────────────
X_train_tfidf   = X_tfidf[idx_train]
X_test_tfidf    = X_tfidf[idx_test]

# ── Entrenar modelos (solo LR, no necesita GPU) ────────────────
print("\nEntrenando modelos...")

# BERT grande + TF-IDF
X_train_m1 = hstack([X_train_tfidf, csr_matrix(X_bert_large[idx_train])])
X_test_m1  = hstack([X_test_tfidf,  csr_matrix(X_bert_large[idx_test])])
clf_m1     = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))
clf_m1.fit(X_train_m1, y_train)
pred_m1    = clf_m1.predict_proba(X_test_m1)
print(f"BERT grande + TF-IDF: {roc_auc_score(y_test, pred_m1, average='macro'):.4f}")

# RoBERTa + TF-IDF
X_train_rob = hstack([X_train_tfidf, csr_matrix(X_roberta[idx_train])])
X_test_rob  = hstack([X_test_tfidf,  csr_matrix(X_roberta[idx_test])])
clf_rob     = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))
clf_rob.fit(X_train_rob, y_train)
pred_rob    = clf_rob.predict_proba(X_test_rob)
print(f"RoBERTa + TF-IDF:     {roc_auc_score(y_test, pred_rob, average='macro'):.4f}")

# E5 + TF-IDF
X_train_e5 = hstack([X_train_tfidf, csr_matrix(X_e5[idx_train])])
X_test_e5  = hstack([X_test_tfidf,  csr_matrix(X_e5[idx_test])])
clf_e5     = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))
clf_e5.fit(X_train_e5, y_train)
pred_e5    = clf_e5.predict_proba(X_test_e5)
print(f"E5 + TF-IDF:          {roc_auc_score(y_test, pred_e5, average='macro'):.4f}")

# ── Optimizar ensemble ─────────────────────────────────────────
print("\nOptimizando ensemble...")
preds_lista   = [pred_m1, pred_rob, pred_e5]
nombres_lista = ['BERT grande + TF-IDF', 'RoBERTa + TF-IDF', 'E5 + TF-IDF']

def neg_roc(weights):
    weights = np.abs(weights) / np.abs(weights).sum()
    pred_ensemble = sum(w * p for w, p in zip(weights, preds_lista))
    return -roc_auc_score(y_test, pred_ensemble, average='macro')

mejor_roc       = 0
mejor_resultado = None
for _ in range(20):
    w0     = np.random.dirichlet(np.ones(3))
    result = minimize(neg_roc, w0, method='Nelder-Mead',
                     options={'maxiter': 1000, 'xatol': 1e-6})
    if -result.fun > mejor_roc:
        mejor_roc       = -result.fun
        mejor_resultado = result

pesos_optimos = np.abs(mejor_resultado.x)
pesos_optimos = pesos_optimos / pesos_optimos.sum()

print("\n══ Pesos óptimos ══")
for nombre, peso in sorted(zip(nombres_lista, pesos_optimos), key=lambda x: x[1], reverse=True):
    print(f"{nombre:<25} {peso:.4f}")
print(f"\nROC ensemble: {mejor_roc:.4f}")

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from scipy.sparse import hstack, csr_matrix
from scipy.optimize import minimize

# ── Cargar datos ───────────────────────────────────────────────
dataTraining = pd.read_csv('https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTraining.zip', encoding='UTF-8', index_col=0)
dataTesting  = pd.read_csv('https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTesting.zip',  encoding='UTF-8', index_col=0)

# ── Limpieza ───────────────────────────────────────────────────
dataTraining1 = dataTraining.drop_duplicates(subset=['title', 'year'])
dataTraining1 = dataTraining1[dataTraining1['plot'].str.split().str.len() >= 10]
dataTraining1['genres'] = dataTraining1['genres'].map(lambda x: eval(x) if isinstance(x, str) else x)
print(f"Shape: {dataTraining1.shape}")

# ── Etiquetas ──────────────────────────────────────────────────
le       = MultiLabelBinarizer()
y_genres = le.fit_transform(dataTraining1['genres'])

# ── Split ──────────────────────────────────────────────────────
plots = dataTraining1['plot'].tolist()
idx_train, idx_test = train_test_split(range(len(plots)), test_size=0.33, random_state=42)
y_train = y_genres[idx_train]
y_test  = y_genres[idx_test]
print(f"Train: {len(idx_train)} | Test: {len(idx_test)}")

# ── Cargar embeddings guardados ────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

X_bert_large = np.load('/content/drive/MyDrive/ML/X_bert_large.npy')
X_roberta    = np.load('/content/drive/MyDrive/ML/X_roberta.npy')
X_e5         = np.load('/content/drive/MyDrive/ML/X_e5.npy')
print(f"Embeddings cargados ✓")

# ── TF-IDF ─────────────────────────────────────────────────────
vect_tfidf = TfidfVectorizer(max_features=10000, stop_words='english')
X_tfidf    = vect_tfidf.fit_transform(dataTraining1['plot']).astype(np.float32)

# ── Splits ─────────────────────────────────────────────────────
X_train_tfidf   = X_tfidf[idx_train]
X_test_tfidf    = X_tfidf[idx_test]

# ── Entrenar modelos (solo LR, no necesita GPU) ────────────────
print("\nEntrenando modelos...")

# BERT grande + TF-IDF
X_train_m1 = hstack([X_train_tfidf, csr_matrix(X_bert_large[idx_train])])
X_test_m1  = hstack([X_test_tfidf,  csr_matrix(X_bert_large[idx_test])])
clf_m1     = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))
clf_m1.fit(X_train_m1, y_train)
pred_m1    = clf_m1.predict_proba(X_test_m1)
print(f"BERT grande + TF-IDF: {roc_auc_score(y_test, pred_m1, average='macro'):.4f}")

# RoBERTa + TF-IDF
X_train_rob = hstack([X_train_tfidf, csr_matrix(X_roberta[idx_train])])
X_test_rob  = hstack([X_test_tfidf,  csr_matrix(X_roberta[idx_test])])
clf_rob     = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))
clf_rob.fit(X_train_rob, y_train)
pred_rob    = clf_rob.predict_proba(X_test_rob)
print(f"RoBERTa + TF-IDF:     {roc_auc_score(y_test, pred_rob, average='macro'):.4f}")

# E5 + TF-IDF
X_train_e5 = hstack([X_train_tfidf, csr_matrix(X_e5[idx_train])])
X_test_e5  = hstack([X_test_tfidf,  csr_matrix(X_e5[idx_test])])
clf_e5     = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))
clf_e5.fit(X_train_e5, y_train)
pred_e5    = clf_e5.predict_proba(X_test_e5)
print(f"E5 + TF-IDF:          {roc_auc_score(y_test, pred_e5, average='macro'):.4f}")

# ── Optimizar ensemble ─────────────────────────────────────────
print("\nOptimizando ensemble...")
preds_lista   = [pred_m1, pred_rob, pred_e5]
nombres_lista = ['BERT grande + TF-IDF', 'RoBERTa + TF-IDF', 'E5 + TF-IDF']

def neg_roc(weights):
    weights = np.abs(weights) / np.abs(weights).sum()
    pred_ensemble = sum(w * p for w, p in zip(weights, preds_lista))
    return -roc_auc_score(y_test, pred_ensemble, average='macro')

mejor_roc       = 0
mejor_resultado = None
for _ in range(20):
    w0     = np.random.dirichlet(np.ones(3))
    result = minimize(neg_roc, w0, method='Nelder-Mead',
                     options={'maxiter': 1000, 'xatol': 1e-6})
    if -result.fun > mejor_roc:
        mejor_roc       = -result.fun
        mejor_resultado = result

pesos_optimos = np.abs(mejor_resultado.x)
pesos_optimos = pesos_optimos / pesos_optimos.sum()

print("\n══ Pesos óptimos ══")
for nombre, peso in sorted(zip(nombres_lista, pesos_optimos), key=lambda x: x[1], reverse=True):
    print(f"{nombre:<25} {peso:.4f}")
print(f"\nROC ensemble: {mejor_roc:.4f}")

Shape: (7875, 5)
Train: 5276 | Test: 2599
Mounted at /content/drive
Embeddings cargados ✓

Entrenando modelos...
BERT grande + TF-IDF: 0.9180
RoBERTa + TF-IDF:     0.9234
E5 + TF-IDF:          0.9280

Optimizando ensemble...

══ Pesos óptimos ══
E5 + TF-IDF               0.6168
RoBERTa + TF-IDF          0.3142
BERT grande + TF-IDF      0.0690

ROC ensemble: 0.9316


In [3]:
from sentence_transformers import SentenceTransformer

# ── Embeddings para dataTesting ────────────────────────────────
print("Generando embeddings para dataTesting...")
plots_kaggle = dataTesting['plot'].tolist()

X_tfidf_kaggle = vect_tfidf.transform(dataTesting['plot']).astype(np.float32)

# BERT grande
bert_large_model    = SentenceTransformer('all-mpnet-base-v2')
X_bert_large_kaggle = bert_large_model.encode(plots_kaggle, batch_size=64, show_progress_bar=True)
print(f"BERT grande kaggle: {X_bert_large_kaggle.shape}")

# RoBERTa
roberta_model    = SentenceTransformer('all-roberta-large-v1')
X_roberta_kaggle = roberta_model.encode(plots_kaggle, batch_size=64, show_progress_bar=True)
print(f"RoBERTa kaggle: {X_roberta_kaggle.shape}")

# E5
e5_model    = SentenceTransformer('intfloat/e5-large-v2')
X_e5_kaggle = e5_model.encode(
    ['passage: ' + p for p in plots_kaggle],
    batch_size=32, show_progress_bar=True
)
print(f"E5 kaggle: {X_e5_kaggle.shape}")

# ── Predicciones finales ───────────────────────────────────────
pred_kaggle_m1  = clf_m1.predict_proba(hstack([X_tfidf_kaggle, csr_matrix(X_bert_large_kaggle)]))
pred_kaggle_rob = clf_rob.predict_proba(hstack([X_tfidf_kaggle, csr_matrix(X_roberta_kaggle)]))
pred_kaggle_e5  = clf_e5.predict_proba(hstack([X_tfidf_kaggle, csr_matrix(X_e5_kaggle)]))

pred_final = (pesos_optimos[0] * pred_kaggle_e5 +
              pesos_optimos[1] * pred_kaggle_rob +
              pesos_optimos[2] * pred_kaggle_m1)

# ── Guardar CSV ────────────────────────────────────────────────
cols = ['p_Action', 'p_Adventure', 'p_Animation', 'p_Biography', 'p_Comedy',
        'p_Crime', 'p_Documentary', 'p_Drama', 'p_Family', 'p_Fantasy',
        'p_Film-Noir', 'p_History', 'p_Horror', 'p_Music', 'p_Musical',
        'p_Mystery', 'p_News', 'p_Romance', 'p_Sci-Fi', 'p_Short',
        'p_Sport', 'p_Thriller', 'p_War', 'p_Western']

res = pd.DataFrame(pred_final, index=dataTesting.index, columns=cols)
res.to_csv('/content/drive/MyDrive/ML/pred_ensemble_final.csv', index_label='ID')
print(f"\nPredicciones guardadas!")
print(f"Shape: {res.shape}")
print(res.head())

Generando embeddings para dataTesting...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Batches:   0%|          | 0/106 [00:00<?, ?it/s]

E5 kaggle: (3383, 1024)

Predicciones guardadas!
Shape: (3383, 24)
   p_Action  p_Adventure  p_Animation  p_Biography  p_Comedy   p_Crime  \
1  0.046881     0.052229     0.010376     0.021364  0.240740  0.040847   
4  0.076006     0.045594     0.005936     0.260481  0.072444  0.453384   
5  0.058863     0.005285     0.001911     0.063672  0.135620  0.891792   
6  0.046217     0.037440     0.002538     0.024915  0.177693  0.029078   
7  0.033559     0.021360     0.013709     0.007076  0.148806  0.072349   

   p_Documentary   p_Drama  p_Family  p_Fantasy  ...  p_Musical  p_Mystery  \
1       0.005146  0.485947  0.015632   0.140967  ...   0.026780   0.129173   
4       0.085080  0.886433  0.018242   0.007997  ...   0.021413   0.027189   
5       0.010057  0.788180  0.004678   0.010435  ...   0.002971   0.389090   
6       0.004024  0.734406  0.006809   0.007916  ...   0.009308   0.038510   
7       0.003953  0.113683  0.026474   0.210248  ...   0.008396   0.326039   

     p_News  p_Roma